# 01 - Historical NFL Data Collection

## Goal

This notebook collects every dataset required to build the NFL Season Projection Model.

Rather than downloading spreadsheets manually, every dataset is collected programmatically so the project can be reproduced and updated in future seasons.

## Data Sources

- nflverse
- NFL schedules
- Team rosters
- Coaching information
- Play-by-play data
- Team statistics
- Advanced metrics

## Output

Processed datasets are saved into the `/data` directory for use throughout the project.

In [16]:
import pandas as pd
import numpy as np
import polars as pl

from pathlib import Path

## Project Directory

In [17]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

ROSTER_DIR = DATA_DIR / "roster"

COACHING_DIR = DATA_DIR / "coaching"

SCHEDULE_DIR = DATA_DIR / "schedules"

print(PROJECT_ROOT)

c:\Users\efriedman\Desktop\NFL-Season-Projections


## Historical Seasons

The first version of the model uses NFL data from 2015 through 2025.

This provides enough recent history to identify patterns while keeping the data relevant to the modern NFL. Older seasons may be added later for specific coaching, roster, and rule-change analysis.

In [18]:
SEASONS = list(range(2015, 2026))

print(SEASONS)
print(f"Number of seasons: {len(SEASONS)}")

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Number of seasons: 11


In [19]:
import nflreadpy as nfl

print("nflreadpy loaded")

nflreadpy loaded


## Test Play-by-Play Download

Before collecting the full historical dataset, one season is downloaded to confirm the data source, notebook environment, and file structure are working correctly.

In [20]:
pbp_test = nfl.load_pbp(seasons=[2025])

print(type(pbp_test))
print(pbp_test.shape)

<class 'polars.dataframe.frame.DataFrame'>
(48771, 372)


In [21]:
# Number of rows and columns
print(f"Rows: {pbp_test.height:,}")
print(f"Columns: {pbp_test.width}")

print("\nFirst 25 columns:\n")

print(pbp_test.columns[:25])

Rows: 48,771
Columns: 372

First 25 columns:

['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team', 'season_type', 'week', 'posteam', 'posteam_type', 'defteam', 'side_of_field', 'yardline_100', 'game_date', 'quarter_seconds_remaining', 'half_seconds_remaining', 'game_seconds_remaining', 'game_half', 'quarter_end', 'drive', 'sp', 'qtr', 'down', 'goal_to_go', 'time', 'yrdln']


In [22]:
pbp_test.head()

play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,side_of_field,yardline_100,game_date,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,game_half,quarter_end,drive,sp,qtr,down,goal_to_go,time,yrdln,ydstogo,ydsnet,desc,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,…,home_coach,away_coach,stadium_id,game_stadium,aborted_play,success,passer,passer_jersey_number,rusher,rusher_jersey_number,receiver,receiver_jersey_number,pass,rush,first_down,special,play,passer_id,rusher_id,receiver_id,name,jersey_number,id,fantasy_player_name,fantasy_player_id,fantasy,fantasy_id,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
f64,str,str,str,str,str,i32,str,str,str,str,f64,str,f64,f64,f64,str,f64,f64,f64,f64,f64,i32,str,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,str,…,str,str,str,str,f64,f64,str,i32,str,i32,str,i32,f64,f64,f64,f64,f64,str,str,str,str,i32,str,str,str,str,str,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64
1.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,null,null,null,null,null,"""2025-09-07""",900.0,1800.0,3600.0,"""Half1""",0.0,null,0.0,1.0,null,0,"""15:00""","""NO 35""",0.0,null,"""GAME""",null,null,0.0,0.0,null,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,null,null,null,null,0.0,0.0,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,0.0,0.0,-0.0,null,null,null,null,null,null,null
40.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""NO""",35.0,"""2025-09-07""",900.0,1800.0,3600.0,"""Half1""",0.0,1.0,0.0,1.0,null,0,"""15:00""","""NO 35""",0.0,2.0,"""19-B.Grupe kicks 65 yards from…","""kickoff""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,null,null,null,null,0.0,0.0,0.0,1.0,0.0,null,null,null,null,null,null,null,null,null,null,0.0,0.0,-0.3527,null,null,null,null,null,null,null
63.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",78.0,"""2025-09-07""",896.0,1796.0,3596.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0,"""14:56""","""ARI 22""",10.0,2.0,"""(14:56) 6-J.Conner right tackl…","""run""",3.0,0.0,0.0,0.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,null,null,"""J.Conner""",6,null,null,0.0,1.0,0.0,0.0,1.0,null,"""00-0033553""",null,"""J.Conner""",6,"""00-0033553""","""J.Conner""","""00-0033553""","""J.Conner""","""00-0033553""",0.0,0.0,-0.190052,null,null,null,null,null,0.511128,-51.112807
85.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",75.0,"""2025-09-07""",858.0,1758.0,3558.0,"""Half1""",0.0,1.0,0.0,1.0,2.0,0,"""14:18""","""ARI 25""",7.0,2.0,"""(14:18) (Shotgun) 1-K.Murray p…","""pass""",11.0,1.0,0.0,1.0,0.0,0.0,0.0,"""short""",…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,1.0,"""K.Murray""",1,null,null,"""T.McBride""",85,1.0,0.0,1.0,0.0,1.0,"""00-0035228""",null,"""00-0037744""","""K.Murray""",1,"""00-0035228""","""T.McBride""","""00-0037744""","""T.McBride""","""00-0037744""",1.0,0.0,1.31734,0.939998,4.750889,3,0.666726,0.43911,0.66894,33.105969
115.0,"""2025_01_ARI_NO""","""2025090705""","""NO""","""ARI""","""REG""",1,"""ARI""","""away""","""NO""","""ARI""",64.0,"""2025-09-07""",820.0,1720.0,3520.0,"""Half1""",0.0,1.0,0.0,1.0,1.0,0,"""13:40""","""ARI 36""",10.0,2.0,"""(13:40) 1-K.Murray sacked at A…","""pass""",-11.0,0.0,0.0,1.0,0.0,0.0,0.0,null,…,"""Kellen Moore""","""Jonathan Gannon""","""NOR00""","""Mercedes-Benz Superdome""",0.0,0.0,"""K.Murray""",1,null,null,null,null,1.0,0.0,0.0,0.0,1.0,"""00-0035228""",null,null,"""K.Murray""",1,"""00-0035228""",null,null,n

# Data Collection Roadmap

The season projection model combines multiple datasets that each answer different football questions.

Rather than relying on a single dataset, the project integrates team performance, player performance, coaching continuity, roster movement, and schedule information into one unified modeling pipeline.

## Planned Datasets

### 1. Play-by-Play Data
Purpose:
- Calculate offensive and defensive efficiency
- EPA
- Success Rate
- Explosive Plays
- Turnovers
- Pressure
- Red Zone Performance

### 2. Team Season Statistics
Purpose:
- Wins
- Losses
- Point Differential
- Home/Road Performance
- One Score Games

### 3. Team Rosters
Purpose:
- Returning starters
- Position groups
- Player continuity

### 4. Player Statistics
Purpose:
- Quarterback projections
- Position group ratings
- Individual improvement

### 5. Coaching Database
Purpose:
- HC/OC/DC changes
- Coordinator continuity
- Previous performance

### 6. Schedule Data
Purpose:
- Opponents
- Home/Away
- Rest
- Travel
- Divisional games

### 7. Transactions
Purpose:
- Free agent additions
- Departures
- Trades
- Draft picks

## Save Raw Play-by-Play Data

The raw dataset is saved locally so future notebooks can use the same source data without downloading it again.

Parquet format is used because it is faster and more space-efficient than CSV while preserving data types.

In [23]:
test_output_path = RAW_DIR / "play_by_play_2025.parquet"

pbp_test.write_parquet(test_output_path)

print(f"Saved to: {test_output_path}")

Saved to: c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\play_by_play_2025.parquet


## Download Historical Play-by-Play Data

The full historical dataset is collected for the 2015–2025 seasons.

Each season is saved as its own Parquet file so the pipeline remains organized, easier to update, and less memory-intensive than combining everything immediately.

In [24]:
for season in SEASONS:
    output_path = RAW_DIR / f"play_by_play_{season}.parquet"

    if output_path.exists():
        print(f"{season}: file already exists — skipping")
        continue

    print(f"{season}: downloading...")

    season_pbp = nfl.load_pbp(seasons=[season])
    season_pbp.write_parquet(output_path)

    print(
        f"{season}: saved {season_pbp.height:,} rows "
        f"and {season_pbp.width} columns"
    )

2015: file already exists — skipping
2016: file already exists — skipping
2017: file already exists — skipping
2018: file already exists — skipping
2019: file already exists — skipping
2020: file already exists — skipping
2021: file already exists — skipping
2022: file already exists — skipping
2023: file already exists — skipping
2024: file already exists — skipping
2025: file already exists — skipping


# Schedule and Game Results Data

Schedule data provides the game-level results and context required to calculate team records, point differential, home and road performance, one-score results, rest, and future schedule difficulty.

The 2015–2025 schedules are collected separately from play-by-play data because each row represents one NFL game rather than one play.

In [25]:
schedules = nfl.load_schedules(seasons=SEASONS)

print(type(schedules))
print(schedules.shape)
print(schedules.columns[:25])

<class 'polars.dataframe.frame.DataFrame'>
(3028, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline']


In [26]:
schedules.select([
    "game_id",
    "season",
    "game_type",
    "week",
    "gameday",
    "away_team",
    "away_score",
    "home_team",
    "home_score",
    "location",
    "overtime",
    "away_rest",
    "home_rest"
]).head(10)

game_id,season,game_type,week,gameday,away_team,away_score,home_team,home_score,location,overtime,away_rest,home_rest
str,i32,str,i32,str,str,i32,str,i32,str,i32,i32,i32
"""2015_01_PIT_NE""",2015,"""REG""",1,"""2015-09-10""","""PIT""",21,"""NE""",28,"""Home""",0,7,7
"""2015_01_IND_BUF""",2015,"""REG""",1,"""2015-09-13""","""IND""",14,"""BUF""",27,"""Home""",0,7,7
"""2015_01_GB_CHI""",2015,"""REG""",1,"""2015-09-13""","""GB""",31,"""CHI""",23,"""Home""",0,7,7
"""2015_01_KC_HOU""",2015,"""REG""",1,"""2015-09-13""","""KC""",27,"""HOU""",20,"""Home""",0,7,7
"""2015_01_CAR_JAX""",2015,"""REG""",1,"""2015-09-13""","""CAR""",20,"""JAX""",9,"""Home""",0,7,7
"""2015_01_CLE_NYJ""",2015,"""REG""",1,"""2015-09-13""","""CLE""",10,"""NYJ""",31,"""Home""",0,7,7
"""2015_01_SEA_STL""",2015,"""REG""",1,"""2015-09-13""","""SEA""",31,"""STL""",34,"""Home""",1,7,7
"""2015_01_MIA_WAS""",2015,"""REG""",1,"""2015-09-13""","""MIA""",17,"""WAS""",10,"""Home""",0,7,7
"""2015_01_NO_ARI""",2015,"""REG""",1,"""2015-09-13""","""NO""",19,"""ARI""",31,"""Home""",0,7,7


In [27]:
schedule_output_path = SCHEDULE_DIR / "nfl_schedules_2015_2025.parquet"

schedules.write_parquet(schedule_output_path)

print(f"Saved {schedules.height:,} games to:")
print(schedule_output_path)

Saved 3,028 games to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\schedules\nfl_schedules_2015_2025.parquet


# Historical Roster Data

Roster data will be used to measure team continuity, returning starters, age, experience, position-group composition, offseason additions and departures, and projected player availability.

Historical rosters are collected separately from play-by-play data because they describe who was on each team rather than what happened on each play.

In [28]:
rosters = nfl.load_rosters(seasons=SEASONS)

print(type(rosters))
print(rosters.shape)
print(rosters.columns[:30])

<class 'polars.dataframe.frame.DataFrame'>
(33195, 36)
['season', 'team', 'position', 'depth_chart_position', 'jersey_number', 'status', 'full_name', 'first_name', 'last_name', 'birth_date', 'height', 'weight', 'college', 'gsis_id', 'espn_id', 'sportradar_id', 'yahoo_id', 'rotowire_id', 'pff_id', 'pfr_id', 'fantasy_data_id', 'sleeper_id', 'years_exp', 'headshot_url', 'ngs_position', 'week', 'game_type', 'status_description_abbr', 'football_name', 'esb_id']


In [29]:
rosters.filter(
    (pl.col("season") == 2025) &
    (pl.col("team") == "GB")
).select([
    "season",
    "week",
    "team",
    "full_name",
    "position",
    "depth_chart_position",
    "status",
    "years_exp"
]).head(20)

season,week,team,full_name,position,depth_chart_position,status,years_exp
i32,i32,str,str,str,str,str,i32
2025,19,"""GB""","""Dante Barnett""","""DL""","""DT""","""DEV""",0
2025,19,"""GB""","""Brandon McManus""","""K""","""K""","""ACT""",12
2025,19,"""GB""","""Rashan Gary""","""DL""","""DE""","""ACT""",6
2025,19,"""GB""","""Matthew Orzech""","""LS""","""LS""","""ACT""",6
2025,19,"""GB""","""Keisean Nixon""","""DB""","""CB""","""ACT""",6
…,…,…,…,…,…,…,…
2025,19,"""GB""","""Zayne Anderson""","""DB""","""FS""","""RES""",4
2025,19,"""GB""","""Nate Hobbs""","""DB""","""CB""","""RES""",4
2025,19,"""GB""","""Micah Parsons""","""LB""","""OLB""","""RES""",4


In [30]:
roster_output_path = ROSTER_DIR / "nfl_rosters_2015_2025.parquet"

rosters.write_parquet(roster_output_path)

print(f"Saved {rosters.height:,} roster records to:")
print(roster_output_path)

Saved 33,195 roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\roster\nfl_rosters_2015_2025.parquet


# Historical Player Statistics

Player statistics provide the individual production data needed for quarterback projections, skill-position evaluation, defensive player analysis, and position-group strength.

Regular-season summaries are collected so each player has one season-level statistical profile per year.

In [31]:
player_stats = nfl.load_player_stats(
    seasons=SEASONS,
    summary_level="reg"
)

print(type(player_stats))
print(player_stats.shape)
print(player_stats.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(21377, 143)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'season_type', 'recent_team', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost']


In [32]:
player_stats.filter(
    (pl.col("season") == 2025) &
    (pl.col("position") == "QB")
).select([
    "player_display_name",
    "recent_team",
    "games",
    "attempts",
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "sacks_suffered",
    "passing_epa",
    "passing_cpoe",
    "carries",
    "rushing_yards",
    "rushing_tds"
]).sort("passing_epa", descending=True).head(15)

player_display_name,recent_team,games,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,passing_epa,passing_cpoe,carries,rushing_yards,rushing_tds
str,str,i32,i32,i32,i32,i32,i32,f64,f64,i32,i32,i32
"""Jimmy Garoppolo""","""LA""",3,0,0,0,0,0,null,null,9,-10,0
"""Jarrett Stidham""","""DEN""",1,0,0,0,0,0,null,null,1,-1,0
"""Adrian Martinez""","""SF""",1,0,0,0,0,0,null,null,1,-1,0
"""Jalen Milroe""","""SEA""",3,0,0,0,0,0,null,null,3,4,0
"""Drake Maye""","""NE""",17,492,4394,31,8,47,165.161542,10.781275,103,450,4
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Josh Allen""","""BUF""",16,460,3668,25,10,40,71.326336,3.514859,112,579,14
"""Patrick Mahomes""","""KC""",14,502,3587,22,11,34,68.246941,0.336415,64,422,5
"""Daniel Jones""","""IND""",13,384,3101,19,8,22,65.255515,2.297142,45,164,5


In [33]:
player_stats_output_path = RAW_DIR / "player_stats_2015_2025.parquet"

player_stats.write_parquet(player_stats_output_path)

print(f"Saved {player_stats.height:,} player-season records to:")
print(player_stats_output_path)

Saved 21,377 player-season records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\player_stats_2015_2025.parquet


# Historical Player Participation Data

Participation data helps distinguish roster membership from actual on-field involvement.

This information will later support measures of returning production, starter continuity, position group stability, roster turnover, and player availability. It is particularly important for evaluating offensive line and defensive front continuity, where traditional box score statistics do not fully represent a player's contribution.

In [35]:
PARTICIPATION_SEASONS = list(range(2016, 2026))

participation = nfl.load_participation(
    seasons=PARTICIPATION_SEASONS
)

print(type(participation))
print(participation.shape)
print(participation.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(478989, 26)
['nflverse_game_id', 'old_game_id', 'play_id', 'possession_team', 'offense_formation', 'offense_personnel', 'defenders_in_box', 'defense_personnel', 'number_of_pass_rushers', 'players_on_play', 'offense_players', 'defense_players', 'n_offense', 'n_defense', 'ngs_air_yards', 'time_to_throw', 'was_pressure', 'route', 'defense_man_zone_type', 'defense_coverage_type', 'offense_names', 'defense_names', 'offense_positions', 'defense_positions', 'offense_numbers', 'defense_numbers']


In [36]:
participation.select([
    "nflverse_game_id",
    "play_id",
    "possession_team",
    "offense_formation",
    "offense_personnel",
    "defenders_in_box",
    "defense_personnel",
    "number_of_pass_rushers",
    "time_to_throw",
    "was_pressure",
    "offense_positions",
    "defense_positions"
]).head(10)

nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,time_to_throw,was_pressure,offense_positions,defense_positions
str,f64,str,str,str,i32,str,i32,f64,bool,str,str
"""2016_01_CAR_DEN""",1.0,"""""",null,null,null,null,null,null,null,null,null
"""2016_01_CAR_DEN""",36.0,"""CAR""",null,null,null,null,null,null,null,null,null
"""2016_01_CAR_DEN""",51.0,"""DEN""","""SINGLEBACK""","""1 RB, 1 TE, 3 WR""",6,"""4 DL, 2 LB, 5 DB""",4,2.323,false,null,null
"""2016_01_CAR_DEN""",75.0,"""DEN""","""I_FORM""","""6 OL, 2 RB, 0 TE, 2 WR""",8,"""4 DL, 3 LB, 4 DB""",6,2.893,true,null,null
"""2016_01_CAR_DEN""",97.0,"""DEN""","""SINGLEBACK""","""1 RB, 1 TE, 3 WR""",7,"""4 DL, 2 LB, 5 DB""",3,2.556,false,null,null
"""2016_01_CAR_DEN""",119.0,"""DEN""","""SHOTGUN""","""1 RB, 1 TE, 3 WR""",6,"""4 DL, 2 LB, 5 DB""",4,4.59,false,null,null
"""2016_01_CAR_DEN""",143.0,"""DEN""","""SINGLEBACK""","""1 RB, 0 TE, 4 WR""",6,"""4 DL, 2 LB, 5 DB""",5,1.502,false,null,null
"""2016_01_CAR_DEN""",167.0,"""DEN""","""I_FORM""","""2 RB, 1 TE, 2 WR""",7,"""4 DL, 3 LB, 4 DB""",null,null,null,null,null
"""2016_01_CAR_DEN""",188.0,"""DEN""","""SINGLEBACK""","""1 RB, 1 TE, 3 WR""",6,"""4 DL, 2 LB, 5 DB""",null,null,null,null,null


In [37]:
participation_output_path = RAW_DIR / "participation_2016_2025.parquet"

participation.write_parquet(participation_output_path)

print(f"Saved {participation.height:,} participation records to:")
print(participation_output_path)

Saved 478,989 participation records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\participation_2016_2025.parquet


# Historical Snap Count Data

Snap counts measure how frequently individual players were actually on the field.

This information will later help quantify returning production, starter continuity, position-group stability, roster turnover, and the importance of player additions and departures.

Snap counts are particularly valuable for offensive-line and defensive evaluation because many important contributors are not adequately represented by traditional box-score statistics.

In [38]:
snap_counts = nfl.load_snap_counts(seasons=SEASONS)

print(type(snap_counts))
print(snap_counts.shape)
print(snap_counts.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(276948, 16)
['game_id', 'pfr_game_id', 'season', 'game_type', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']


In [39]:
snap_counts.filter(
    (pl.col("season") == 2025) &
    (pl.col("team") == "GB") &
    (pl.col("position").is_in(["LT", "LG", "C", "RG", "RT", "OL"]))
).select([
    "week",
    "player",
    "position",
    "team",
    "opponent",
    "offense_snaps",
    "offense_pct"
]).sort([
    "week",
    "offense_snaps"
], descending=[False, True]).head(25)

week,player,position,team,opponent,offense_snaps,offense_pct
i32,str,str,str,str,f64,f64
1,"""Elgton Jenkins""","""C""","""GB""","""DET""",48.0,1.0
1,"""Zach Tom""","""OL""","""GB""","""DET""",30.0,0.62
2,"""Elgton Jenkins""","""C""","""GB""","""WAS""",68.0,1.0
3,"""Elgton Jenkins""","""C""","""GB""","""CLE""",65.0,1.0
3,"""Zach Tom""","""OL""","""GB""","""CLE""",1.0,0.02
…,…,…,…,…,…,…
13,"""Jacob Monk""","""C""","""GB""","""DET""",0.0,0.0
14,"""Zach Tom""","""OL""","""GB""","""CHI""",53.0,1.0
14,"""Jacob Monk""","""C""","""GB""","""CHI""",0.0,0.0


In [40]:
snap_counts_output_path = RAW_DIR / "snap_counts_2015_2025.parquet"

snap_counts.write_parquet(snap_counts_output_path)

print(f"Saved {snap_counts.height:,} player-game snap records to:")
print(snap_counts_output_path)

Saved 276,948 player-game snap records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\snap_counts_2015_2025.parquet


# Historical Depth Chart Data

Depth chart data provides information about player roles and position group hierarchy within each team.

This dataset will later help identify starters, backups, roster competitions, positional depth, and changes in expected playing roles from one season to the next.

In [41]:
depth_charts = nfl.load_depth_charts(seasons=SEASONS)

print(type(depth_charts))
print(depth_charts.shape)
print(depth_charts.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(923447, 26)
['season', 'club_code', 'week', 'game_type', 'depth_team', 'last_name', 'first_name', 'football_name', 'formation', 'gsis_id', 'jersey_number', 'position', 'elias_id', 'depth_position', 'full_name', 'dt', 'team', 'player_name', 'espn_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']


In [47]:
depth_charts.filter(
    pl.col("season").is_null()
).select([
    "dt",
    "team",
    "club_code",
    "week",
    "game_type",
    "player_name",
    "position",
    "depth_position",
    "pos_grp",
    "pos_name",
    "pos_rank"
]).head(20)

dt,team,club_code,week,game_type,player_name,position,depth_position,pos_grp,pos_name,pos_rank
str,str,str,i32,str,str,str,str,str,str,i32
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Josh Sweat""",null,null,"""Base 4-3 D""","""Left Defensive End""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Roy Lopez""",null,null,"""Base 4-3 D""","""Left Defensive Tackle""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Walter Nolen III""",null,null,"""Base 4-3 D""","""Right Defensive Tackle""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Darius Robinson""",null,null,"""Base 4-3 D""","""Right Defensive End""",1
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Cody Simon""",null,null,"""Base 4-3 D""","""Weakside Linebacker""",1
…,…,…,…,…,…,…,…,…,…,…
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Dante Stills""",null,null,"""Base 4-3 D""","""Right Defensive End""",2
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Owen Pappoe""",null,null,"""Base 4-3 D""","""Weakside Linebacker""",2
"""2026-03-14T07:32:09Z""","""ARI""",null,null,null,"""Austin Keys""",null,null,"""Base 4-3 D""","""Middle Linebacker""",2


In [48]:
depth_chart_output_path = RAW_DIR / "depth_charts_raw.parquet"

depth_charts.write_parquet(depth_chart_output_path)

print(f"Saved {depth_charts.height:,} depth chart records to:")
print(depth_chart_output_path)

Saved 923,447 depth chart records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\depth_charts_raw.parquet


# Historical Draft Pick Data

Draft data provides information about incoming rookies and how teams invested draft capital across positions.

This dataset will later support rookie impact estimates, roster turnover analysis, position-group investment, and team-building evaluation.

In [49]:
draft_picks = nfl.load_draft_picks(seasons=SEASONS)

print(type(draft_picks))
print(draft_picks.shape)
print(draft_picks.columns[:35])

<class 'polars.dataframe.frame.DataFrame'>
(2821, 36)
['season', 'round', 'pick', 'team', 'gsis_id', 'pfr_player_id', 'cfb_player_id', 'pfr_player_name', 'hof', 'position', 'category', 'side', 'college', 'age', 'to', 'allpro', 'probowls', 'seasons_started', 'w_av', 'car_av', 'dr_av', 'games', 'pass_completions', 'pass_attempts', 'pass_yards', 'pass_tds', 'pass_ints', 'rush_atts', 'rush_yards', 'rush_tds', 'receptions', 'rec_yards', 'rec_tds', 'def_solo_tackles', 'def_ints']


In [50]:
draft_picks.filter(
    pl.col("season") == 2025
).select([
    "round",
    "pick",
    "team",
    "pfr_player_name",
    "position",
    "category",
    "side",
    "college",
    "age"
]).sort("pick").head(20)

round,pick,team,pfr_player_name,position,category,side,college,age
i32,i32,str,str,str,str,str,str,i32
1,1,"""TEN""","""Cam Ward""","""QB""","""QB""","""O""","""Miami (FL)""",23
1,2,"""JAX""","""Travis Hunter""","""WR""","""WR""","""O""","""Colorado""",22
1,3,"""NYG""","""Abdul Carter""","""DE""","""DL""","""D""","""Penn St.""",21
1,4,"""NWE""","""Will Campbell""","""OT""","""OL""","""O""","""LSU""",21
1,5,"""CLE""","""Mason Graham""","""DT""","""DL""","""D""","""Michigan""",22
…,…,…,…,…,…,…,…,…
1,16,"""ARI""","""Walter Nolen""","""DT""","""DL""","""D""","""Mississippi""",21
1,17,"""CIN""","""Shemar Stewart""","""DE""","""DL""","""D""","""Texas A&M""",21
1,18,"""SEA""","""Grey Zabel""","""OT""","""OL""","""O""","""North Dakota St.""",23


In [51]:
draft_output_path = RAW_DIR / "draft_picks_2015_2025.parquet"

draft_picks.write_parquet(draft_output_path)

print(f"Saved {draft_picks.height:,} draft picks to:")
print(draft_output_path)

Saved 2,821 draft picks to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\raw\draft_picks_2015_2025.parquet


# Historical Injury Data

Injury data provides context about player availability and missed time across NFL seasons.

This dataset will later support durability measures, returning-player adjustments, roster continuity, position-group availability, and estimates of how injuries affected team performance.

Injuries will be treated as context rather than assumed to be perfectly predictable.